# SchoolBridge — Sentence Grouping 자체화 PoC

**진짜 목표**: 가정통신문에서 **의미 묶음별 sentence_list** 추출 (학교 양식·표 구조 무관, 변형 0)

## 핵심 frame

단순 BIO 토큰 분류 X. **각 토큰에 `sentence_id` 정수 부여** → 같은 id 토큰들이 한 의미 묶음.

**예시 (구강검진 표)**:
```
토큰:        [진심담은치과의원, (031)821-8828, 평화로 636, ..., 청담i치과의원, (031)837-1111, ...]
sentence_id: [      1,             1,            1,       ...,     2,              2,         ...]
```

→ 시각적으로 행 단위 흩어져 있어도 모델이 학습으로 같은 묶음으로 결합. **룰·정렬 0**.

## 출력 — Span-based sentence_list

```python
[
  "진심담은치과의원 (031)821-8828 평화로 636 ... 평일 9:30~18:30",
  "청담i치과의원 (031)837-1111 신흥로240번길 ... 평일 10:00~18:00",
  ...
]
```

원문 토큰 그대로 join (생성 X, 변형 0).

## 모델

`microsoft/layoutxlm-base` + token classification head (num_labels = `MAX_SENT_ID`). 토큰별로 sentence_id (0~49) 예측.

## 1. 환경 + 설치

In [ ]:
!pip install -q transformers sentencepiece pdfplumber pymupdf pillow
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
!apt-get install -y -q fonts-nanum libreoffice > /dev/null 2>&1
!fc-cache -fv > /dev/null 2>&1
print("설치 완료 — 런타임 재시작 필요할 수 있음 (UI: 런타임 → 세션 다시 시작)")

In [ ]:
import torch
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
import detectron2
print("detectron2:", detectron2.__version__)
from transformers import LayoutXLMProcessor, LayoutLMv2ForTokenClassification
print("LayoutXLM import OK")

## 2. 파일 업로드

In [ ]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
uploaded_paths = [Path(name) for name in uploaded.keys()]
for p in uploaded_paths:
    print(f"  {p.name}: {p.stat().st_size // 1024} KB")

## 3. HWP → PDF (LibreOffice)

In [ ]:
import subprocess

converted = []
for p in uploaded_paths:
    if p.suffix.lower() == ".hwp":
        print(f"변환: {p.name}")
        r = subprocess.run(
            ["libreoffice", "--headless", "--convert-to", "pdf", str(p)],
            capture_output=True, text=True, timeout=60,
        )
        pdf_path = p.with_suffix(".pdf")
        if r.returncode == 0 and pdf_path.exists():
            print(f"  → {pdf_path.name}")
            converted.append(pdf_path)
        else:
            print(f"  ⚠️ 실패: {r.stderr[:200]}")
uploaded_paths.extend(converted)

## 4. 텍스트 + bbox 추출

글자 위치만 추출. 이후 sentence_id 부여는 모델.

In [ ]:
from dataclasses import dataclass
from typing import List, Tuple, Optional

@dataclass
class TextSpan:
    text: str
    bbox: Tuple[float, float, float, float]
    page: int = 0

In [ ]:
import zipfile
import xml.etree.ElementTree as ET

HP_NS = "http://www.hancom.co.kr/hwpml/2011/paragraph"

def extract_hwpx(path: Path) -> List[TextSpan]:
    """HWPX에서 토큰 추출 — cell_id 없이 단어 단위만"""
    spans: List[TextSpan] = []
    with zipfile.ZipFile(path) as z:
        section_files = sorted(n for n in z.namelist()
                                if n.startswith("Contents/section") and n.endswith(".xml"))
        for sf in section_files:
            root = ET.fromstring(z.read(sf))
            # 모든 paragraph 텍스트 추출 — 표 안/밖 구분 안 함 (모델이 학습)
            for p_idx, para in enumerate(root.iter(f"{{{HP_NS}}}p")):
                texts = [t.text for t in para.iter() if t.text]
                for word in " ".join(texts).split():
                    if not word.strip():
                        continue
                    # bbox는 paragraph 순서 기반 시뮬레이션 (HWPX엔 픽셀 좌표 없음)
                    y = min(0.99, p_idx * 0.025)
                    bbox = (0.0, y, 1.0, min(1.0, y + 0.02))
                    spans.append(TextSpan(text=word, bbox=bbox, page=0))
    return spans

In [ ]:
import pdfplumber

def extract_pdf(path: Path) -> List[TextSpan]:
    """PDF에서 단어 + bbox 추출 — cell_id 없이"""
    spans: List[TextSpan] = []
    with pdfplumber.open(path) as pdf:
        for page_idx, page in enumerate(pdf.pages):
            page_w, page_h = page.width, page.height
            words = page.extract_words(
                use_text_flow=True,
                keep_blank_chars=False,
                x_tolerance=3,
                y_tolerance=3,
            )
            for w in words:
                bbox = (w["x0"] / page_w, w["top"] / page_h,
                        w["x1"] / page_w, w["bottom"] / page_h)
                spans.append(TextSpan(text=w["text"], bbox=bbox, page=page_idx))
    return spans

In [ ]:
all_spans = {}
for p in uploaded_paths:
    ext = p.suffix.lower()
    if ext == ".hwpx":
        spans = extract_hwpx(p)
    elif ext == ".pdf":
        spans = extract_pdf(p)
    else:
        continue
    all_spans[p.name] = spans
    print(f"📄 {p.name}: {len(spans)} tokens")
    for s in spans[:15]:
        print(f"    {s.text!r}")

## 5. 페이지 이미지 렌더링

In [ ]:
import fitz
from PIL import Image

def render_pdf_pages(path: Path, dpi: int = 150) -> List[Image.Image]:
    images = []
    doc = fitz.open(path)
    for page in doc:
        pix = page.get_pixmap(dpi=dpi)
        images.append(Image.frombytes("RGB", (pix.width, pix.height), pix.samples))
    doc.close()
    return images

pdf_paths = [p for p in uploaded_paths if p.suffix.lower() == ".pdf"]
if pdf_paths:
    sample_pdf = pdf_paths[0]
    pdf_pages = render_pdf_pages(sample_pdf, dpi=150)
    print(f"{sample_pdf.name}: {len(pdf_pages)} page(s), {pdf_pages[0].size}")
    pdf_pages[0]

## 6. LayoutXLM 모델 로드 — Sentence Grouping 용

**라벨**: 페이지당 최대 `MAX_SENT_ID = 50` 개의 의미 묶음. 토큰별로 `0~49` 정수 분류.

In [ ]:
from transformers import LayoutXLMProcessor, LayoutLMv2ForTokenClassification
import torch

MODEL_ID = "microsoft/layoutxlm-base"
MAX_SENT_ID = 50  # 페이지당 최대 의미 묶음 수

processor = LayoutXLMProcessor.from_pretrained(MODEL_ID, apply_ocr=False)
model = LayoutLMv2ForTokenClassification.from_pretrained(
    MODEL_ID,
    num_labels=MAX_SENT_ID,
    id2label={i: f"SENT_{i}" for i in range(MAX_SENT_ID)},
    label2id={f"SENT_{i}": i for i in range(MAX_SENT_ID)},
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).eval()
print(f"Model: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params")
print(f"Device: {device}")
print(f"Labels: SENT_0 ~ SENT_{MAX_SENT_ID-1}")

## 7. 입력 변환

In [ ]:
def to_layoutxlm_inputs(image: Image.Image, spans: List[TextSpan], page: int = 0):
    page_spans = [s for s in spans if s.page == page]
    words = [s.text for s in page_spans]
    boxes = []
    for s in page_spans:
        x0, y0, x1, y1 = s.bbox
        boxes.append([
            max(0, min(1000, int(x0 * 1000))),
            max(0, min(1000, int(y0 * 1000))),
            max(0, min(1000, int(x1 * 1000))),
            max(0, min(1000, int(y1 * 1000))),
        ])
    encoded = processor(
        image, words, boxes=boxes,
        return_tensors="pt",
        truncation=True, padding="max_length", max_length=512,
    )
    return encoded, page_spans

## 8. 추론 함수 — Sentence Grouping → sentence_list

**핵심**: 
1. 토큰별 sentence_id 예측
2. 토큰 → 단어 sentence_id (voting)
3. 같은 sentence_id 단어들 그룹화
4. **원문 텍스트 그대로 join** (변형 0)

In [ ]:
from collections import Counter, defaultdict

def predict_sentence_list(image: Image.Image, spans: List[TextSpan]) -> List[str]:
    """페이지 → 의미 묶음별 sentence_list. 텍스트 변형 0."""
    encoded, page_spans = to_layoutxlm_inputs(image, spans, page=0)
    inputs = {k: v.to(device) for k, v in encoded.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    token_preds = outputs.logits.argmax(-1)[0].tolist()
    
    # 토큰 → 단어 sentence_id (voting)
    word_ids = encoded.word_ids() if hasattr(encoded, "word_ids") else None
    word_sids = []
    if word_ids:
        for w_idx in range(len(page_spans)):
            tok_preds = [token_preds[t] for t, w in enumerate(word_ids) if w == w_idx]
            if tok_preds:
                word_sids.append(Counter(tok_preds).most_common(1)[0][0])
            else:
                word_sids.append(-1)
    else:
        word_sids = [-1] * len(page_spans)
    
    # 같은 sentence_id 그룹화
    groups = defaultdict(list)
    first_appearance = {}
    for idx, (span, sid) in enumerate(zip(page_spans, word_sids)):
        if sid >= 0:
            groups[sid].append(span)
            first_appearance.setdefault(sid, idx)
    
    # 그룹 출력 순서 = 각 그룹의 첫 토큰 페이지 등장 순서
    sorted_sids = sorted(groups.keys(), key=lambda s: first_appearance[s])
    
    # 텍스트 join — 원문 그대로 (변형 0)
    sentences = []
    for sid in sorted_sids:
        text = " ".join(s.text for s in groups[sid])
        sentences.append(text)
    
    return sentences

In [ ]:
# 추론 실행 — random init이라 결과는 의미 X. 파이프라인 + 출력 형식 검증.
if pdf_paths and 'pdf_pages' in dir():
    sentences = predict_sentence_list(pdf_pages[0], all_spans[sample_pdf.name])
    print(f"예측 sentence_list ({len(sentences)}개 묶음):")
    for i, s in enumerate(sentences[:10]):
        print(f"  [{i}] {s[:120]}")
    print("\n→ random init이라 그룹화 의미 없음. fine-tune 후 실제 의미 묶음.")

## 9. 라벨링 데이터 형식

**핵심**: 페이지의 토큰들에 의미 묶음 ID 부여. 페이지 내 등장 순서대로 1, 2, 3...

### 구강검진 표 라벨링 예시

In [ ]:
import json

sample_label = {
    "image": "구강검진_p1.png",
    "page_size": [1240, 1753],
    "tokens": [
        # sentence_id=0 — 통신문 상단 발송 정보
        {"text": "발송처", "bbox": [100, 80, 180, 110], "sentence_id": 0},
        {"text": "가능초등학교", "bbox": [200, 80, 320, 110], "sentence_id": 0},
        {"text": "교무실", "bbox": [340, 80, 400, 110], "sentence_id": 0},
        {"text": "031-872-8442", "bbox": [420, 80, 560, 110], "sentence_id": 0},
        
        # sentence_id=1 — 본문 인사말
        {"text": "학부모님", "bbox": [100, 180, 180, 210], "sentence_id": 1},
        {"text": "안녕하십니까?", "bbox": [200, 180, 360, 210], "sentence_id": 1},
        
        # sentence_id=2 — 본문 안내
        {"text": "학교건강검사규칙에", "bbox": [100, 230, 280, 260], "sentence_id": 2},
        {"text": "따라", "bbox": [290, 230, 340, 260], "sentence_id": 2},
        # ... 본문 한 문장 다 같은 id=2
        
        # sentence_id=3 — 1. 검진 대상 항목
        {"text": "1.", "bbox": [100, 400, 130, 430], "sentence_id": 3},
        {"text": "검진", "bbox": [140, 400, 200, 430], "sentence_id": 3},
        {"text": "대상:", "bbox": [210, 400, 280, 430], "sentence_id": 3},
        {"text": "2, 3, 5, 6학년", "bbox": [290, 400, 470, 430], "sentence_id": 3},
        
        # === 표 (진심담은치과 묶음 vs 청담i치과 묶음) ===
        # sentence_id=10 — 진심담은치과의원 정보 묶음 (행 단위 흩어져 있어도 같은 id)
        {"text": "진심담은치과의원", "bbox": [300, 800, 480, 830], "sentence_id": 10},
        {"text": "(031)821-8828", "bbox": [300, 860, 460, 890], "sentence_id": 10},
        {"text": "평화로", "bbox": [300, 920, 360, 950], "sentence_id": 10},
        {"text": "636", "bbox": [370, 920, 410, 950], "sentence_id": 10},
        {"text": "가능역", "bbox": [300, 950, 360, 980], "sentence_id": 10},
        {"text": "112m", "bbox": [370, 950, 420, 980], "sentence_id": 10},
        {"text": "평일", "bbox": [300, 1010, 350, 1040], "sentence_id": 10},
        {"text": "9:30~18:30", "bbox": [360, 1010, 480, 1040], "sentence_id": 10},
        # ... 진심담은치과의 모든 정보가 id=10
        
        # sentence_id=11 — 청담i치과의원 정보 묶음
        {"text": "청담i(아이)치과의원", "bbox": [500, 800, 700, 830], "sentence_id": 11},
        {"text": "(031)837-1111", "bbox": [500, 860, 660, 890], "sentence_id": 11},
        {"text": "신흥로240번길", "bbox": [500, 920, 640, 950], "sentence_id": 11},
        {"text": "21", "bbox": [650, 920, 670, 950], "sentence_id": 11},
        # ... 청담i치과의 모든 정보가 id=11
    ],
}

print(json.dumps(sample_label, ensure_ascii=False, indent=2)[:2500])

**핵심 포인트**:
- 두 치과 정보가 **페이지 시각적으로는 행 단위로 인터리브** (진심담은 전화 → 청담i 전화 → 진심담은 주소 → 청담i 주소 ...)
- 하지만 **sentence_id로는 한 치과 정보가 한 묶음** (id=10 vs id=11)
- 모델이 학습해야 할 패턴 = "같은 열 셀들은 같은 묶음" (이번 양식)
- 다른 학교는 "같은 행이 한 묶음"일 수도 — 학습 데이터로 일반화

**라벨링 가이드**: 페이지 보면서 의미 단위로 묶음. 같은 묶음 토큰들에 같은 정수 id 부여. id 자체는 페이지 내에서만 의미 있음 (페이지 간 무관).

## 10. Fine-tune 학습 코드 스켈레톤

라벨링 JSON 1,000장+ 준비되면 실제 학습. GPU 환경 필요.

In [ ]:
from torch.utils.data import Dataset
from transformers import TrainingArguments, Trainer

class SentenceGroupingDataset(Dataset):
    """라벨링된 통신문 페이지 → LayoutXLM 입력 (sentence_id 라벨)"""
    def __init__(self, annotation_paths, processor, max_sent_id=MAX_SENT_ID):
        self.items = [json.loads(Path(p).read_text(encoding="utf-8")) for p in annotation_paths]
        self.processor = processor
        self.max_sent_id = max_sent_id
    
    def __len__(self):
        return len(self.items)
    
    def __getitem__(self, idx):
        item = self.items[idx]
        image = Image.open(item["image"]).convert("RGB")
        W, H = item["page_size"]
        words = [t["text"] for t in item["tokens"]]
        boxes = [[int(t["bbox"][0]/W*1000), int(t["bbox"][1]/H*1000),
                  int(t["bbox"][2]/W*1000), int(t["bbox"][3]/H*1000)]
                 for t in item["tokens"]]
        # sentence_id를 0 ~ max_sent_id-1 범위로 clip
        word_labels = [min(t["sentence_id"], self.max_sent_id - 1) for t in item["tokens"]]
        
        encoded = self.processor(
            image, words, boxes=boxes, word_labels=word_labels,
            return_tensors="pt", truncation=True,
            padding="max_length", max_length=512,
        )
        return {k: v.squeeze(0) for k, v in encoded.items()}

# 실제 학습 (라벨링 데이터 준비 후):
# train_ds = SentenceGroupingDataset(sorted(Path("data/labeled/train").glob("*.json")), processor)
# val_ds = SentenceGroupingDataset(sorted(Path("data/labeled/val").glob("*.json")), processor)
# 
# args = TrainingArguments(
#     output_dir="./layoutxlm_sentgroup_v1",
#     num_train_epochs=5,
#     per_device_train_batch_size=2,
#     learning_rate=5e-5,
#     warmup_ratio=0.1,
#     evaluation_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     fp16=True,
# )
# trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds)
# trainer.train()
print("학습 스켈레톤 — 라벨링 데이터 준비 후 GPU에서 실행")

## 11. 라벨링 도구 — Label Studio (Relation Extraction 모드)

**셋업**:
```bash
pip install label-studio
label-studio start
```

**프로젝트 설정**:
1. 새 프로젝트 → Computer Vision → "Object Detection with Labels" 또는 "Sequence Labeling" 템플릿
2. 라벨링 UI XML 커스터마이즈 — sentence_id 정수 입력 가능하게
3. 페이지 이미지 + 토큰 bbox 자동 표시 (pre-annotation)
4. 사람이 페이지 보면서:
   - 같은 의미 묶음 토큰들 드래그 또는 클릭 → 같은 id 부여
   - 새 묶음 시작 → 새 id

**소요**: 1장당 2~5분 × 1,000장 = 30~80시간. 팀 분담 또는 외주.

**대안 — 자동 pre-annotation으로 시간 단축**:
- KoELECTRA + KcELECTRA 출력을 seed sentence_id로 자동 부여
- 사람은 검수·수정만 (1장 30초~1분으로 단축)
- 우리가 가진 47,148행 + 15,948행 라벨링 데이터 자산 활용

## 12. 정리

**여기까지 검증된 것**:
1. ✅ HWPX/PDF → 토큰 + bbox 추출 (구조 의미 X, 순수 텍스트 위치)
2. ✅ LayoutXLM 한국어 토큰화 (학년 → 1~2 토큰)
3. ✅ 모델 입력 형식 (image + tokens + bboxes)
4. ✅ 추론 → sentence_list 출력 함수 (변형 0)

**남은 단계**:
1. ⏳ **라벨링** — 페이지별 sentence_id 부여. 핵심: 같은 의미 묶음에 같은 id (시각 위치 무관)
2. ⏳ **Fine-tune** — 1,000장 × 5 epoch (GPU 12~24시간)
3. ⏳ **평가** — Claude 1단계 vs LayoutXLM 동일 통신문에서 sentence_list 비교
4. ⏳ **NCP CPU inference 통합** — LLM 1단계 완전 분리

**핵심**: 룰·정렬 0, 학교·표 양식 무관, 데이터 누적으로 일반화.